# P1-L6｜数据契约与策略报告

> **对应课程**：第 26–30 课  
> **模块**：P1-M6 · 数据契约与策略报告  
> **数据身份**：固定随机种子模拟教学数据，不是真实证券行情。

把数据、信号、基准、风险、成本和稳健性证据组装成可复现报告。

本实验向最终策略报告提供：**数据契约、完整报告与阶段验收**。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 生成最终报告所需的统一证据表

报告同时使用策略、基准、风险、成本和数据契约。缺少任一字段时应视为未完成，而不是留空交付。

In [2]:
from src.quant_core import performance_summary

daily = pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])
close = daily.pivot(index="date", columns="symbol", values="adj_close").sort_index()
asset_return = close["SIM_VALUE"].pct_change(fill_method=None)
benchmark = close["SIM_MARKET"].pct_change(fill_method=None)
signal = (close["SIM_VALUE"].rolling(20).mean() > close["SIM_VALUE"].rolling(60).mean()).astype(float)
position = signal.shift(1).fillna(0.0)
turnover = position.diff().abs().fillna(position.abs())
strategy = position * asset_return - turnover * 0.0005

metrics = performance_summary(strategy, benchmark)
contract = {
    "identity": "synthetic",
    "frequency": "1 business day",
    "timezone": "Asia/Shanghai (teaching convention)",
    "adjustment": "simulated adjusted close",
    "signal_time": "close t",
    "earliest_position_time": "t+1",
    "cost_per_turnover": 0.0005,
}

required_metric_fields = {
    "strategy_annual_return", "benchmark_annual_return",
    "strategy_annual_volatility", "max_drawdown", "beta",
}
assert required_metric_fields.issubset(metrics.index)
assert all(contract.values())
display(metrics.to_frame("value"))
display(pd.Series(contract).to_frame("value"))

,value
strategy_annual_return,0.018241
benchmark_annual_return,-0.012624
strategy_annual_volatility,0.105726
max_drawdown,-0.150885
beta,0.434989
mean_daily_excess,0.000106
observations,755.000000


,value
identity,synthetic
frequency,1 business day
timezone,Asia/Shanghai (teaching convention)
adjustment,simulated adjusted close
signal_time,close t
earliest_position_time,t+1
cost_per_turnover,0.0005


## 写出可归档策略报告

HTML 文件是阶段产物；能力矩阵用于指出仍未通过的实盘、稳健性和工程要求。

In [3]:
metric_rows = "".join(f"<tr><th>{k}</th><td>{v}</td></tr>" for k, v in metrics.items())
contract_rows = "".join(f"<tr><th>{k}</th><td>{v}</td></tr>" for k, v in contract.items())
html = f"""<!doctype html><meta charset='utf-8'>
<title>P1 策略研究报告</title>
<h1>P1 策略研究报告</h1>
<h2>研究假设</h2><p>模拟价值资产的中期趋势是否产生基准和成本后的收益。</p>
<h2>数据契约</h2><table>{contract_rows}</table>
<h2>收益、风险与基准</h2><table>{metric_rows}</table>
<h2>执行与成本</h2><p>收盘形成信号，下一交易日持仓；单位换手成本 5 bps。</p>
<h2>方法边界</h2><p>固定种子模拟行情；未完成参数稳健性、容量、税费、模拟盘和实盘风控。</p>
"""
path = OUTPUT_DIR / "L6_p1_strategy_report.html"
path.write_text(html, encoding="utf-8")
print("已生成:", path)

已生成: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/outputs/L6_p1_strategy_report.html


## 验收

- 关键计算能由小例子或独立公式核对；
- 数据时点、信号时点和成交时点没有混用；
- 结果同时包含金融含义和失败边界；
- 产物能够进入第 29 课策略报告和第 30 课能力矩阵。